# PI III, Aula 08: código da apresentação

Projeto Integrador III. Grupo: Arthur Davino, Lorenzo Louzada e Luis Felipe Ruas.

Este notebook roda, em ordem, todo o código que aparece nos slides e gera as mesmas saídas. Kernel: R (no Colab, Ambiente de execução, Alterar tipo de ambiente de execução, R).

Como usar:
1. Antes de apresentar, use Ambiente de execução, Executar tudo. A célula do cross encoder (Parte 2, item 2.2) instala o modelo e demora alguns minutos, mas só na primeira vez: ela salva `ce_escores.csv` e, se o arquivo já existir, é pulada.
2. Na hora de apresentar, reexecute só as células de cada parte (Luis, Arthur, Lorenzo).
3. Precisa de internet: os dados vêm do GitHub do grupo e o motor08.R do repositório da disciplina.

Mapa dos slides: Parte 1 (Luis) slides 2 a 4, Parte 2 (Arthur) slides 5 a 8, Parte 3 (Lorenzo) slides 9 a 12.

## Parte 1 · Luis: ambiente, corpus e BM25

### 1.1 Carregar o ambiente (slide 4)

In [2]:
REPO <- paste0(
  "https://raw.githubusercontent.com/",
  "luisfeliperuas18-tech/",
  "PI3-LUISFELIPERUAS-LORENZOLOUZADA-ARTHURDAVINO/main/"
)

source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")

cfg <- cfg_aula()
cfg$acentos <- "translit"

corpus <- read.csv(
  paste0(REPO, "estrutura/codigo/corpus/corpus_baixada_santista.csv"),
  fileEncoding = "UTF-8-BOM", stringsAsFactors = FALSE
)
necessidades <- read.csv(
  paste0(REPO, "estrutura/codigo/to%20delete/5-entrega-necessidades.csv"),
  fileEncoding = "UTF-8-BOM", stringsAsFactors = FALSE
)
julgamentos <- read.csv(
  paste0(REPO, "estrutura/codigo/corpus/7-qrels_consolidado_resolvidos.csv"),
  stringsAsFactors = FALSE
)

docs <- setNames(corpus$texto, corpus$id)
consultas <- setNames(
  necessidades$texto_consulta,
  necessidades$consulta
)

ix <- montar(docs, cfg)   # índice do BM25

cat("Documentos:", length(docs), "\n")
cat("Consultas:", length(consultas), "\n")
cat("Julgamentos:", nrow(julgamentos), "\n")

Documentos: 220 
Consultas: 5 
Julgamentos: 86 


### 1.2 Números do corpus (slide 2)

In [3]:
prefixo <- sub("_.*", "", corpus$id)
cat("Santos:", sum(prefixo == "san"), "\n")
cat("São Vicente:", sum(prefixo == "sv"), "\n")
cat("Cubatão:", sum(prefixo == "cub"), "\n")
cat("Termos no vocabulário:", length(ix$vocab), "\n")
cat("Termos por trecho (média):", round(ix$avgdl), "\n")

cat("\nJulgamentos por nota:\n")
print(table(julgamentos$grau))
cat("\nTrechos relevantes (nota 2) por consulta:\n")
print(table(julgamentos$consulta[julgamentos$grau == 2]))
cat("Total de trechos relevantes:", sum(julgamentos$grau == 2), "\n")

Santos: 141 
São Vicente: 43 
Cubatão: 36 
Termos no vocabulário: 4757 
Termos por trecho (média): 92 

Julgamentos por nota:

 0  1  2 
72  5  9 

Trechos relevantes (nota 2) por consulta:

q01 q02 q03 q04 q05 
  2   2   2   1   2 
Total de trechos relevantes: 9 


### 1.3 Divisão das consultas e BM25 puro (slide 3)

Desenvolvimento (q01 a q03) serve para escolher parâmetros. Teste (q04 e q05) não entra na escolha.

In [4]:
qrels <- lapply(unique(julgamentos$consulta), function(q) {
  d <- julgamentos[julgamentos$consulta == q, ]
  setNames(d$grau, d$documento)
})
names(qrels) <- unique(julgamentos$consulta)

dev   <- c("q01", "q02", "q03")
teste <- c("q04", "q05")
ks    <- c(0, 5, 10, 20)

ap_bm25 <- function(q) {
  metricas(names(ranking_bm25(consultas[[q]], ix, cfg)),
           qrels[[q]], cfg$limiar)[["AP"]]
}

map_bm25_dev   <- mean(sapply(dev, ap_bm25))
map_bm25_teste <- mean(sapply(teste, ap_bm25))

cat("BM25 puro, MAP no desenvolvimento:", round(map_bm25_dev, 3), "\n")
cat("BM25 puro, MAP no teste:", round(map_bm25_teste, 3), "\n")

BM25 puro, MAP no desenvolvimento: 0.658 
BM25 puro, MAP no teste: 0.708 


## Parte 2 · Arthur: dois estágios e cross encoder

### 2.1 Pares consulta e trecho (slide 6)

In [5]:
pares <- expand.grid(
  consulta = names(consultas),
  id = names(docs),
  stringsAsFactors = FALSE
)

pares$texto_consulta <- unname(consultas[pares$consulta])
pares$texto_documento <- unname(docs[pares$id])

stopifnot(
  !anyNA(pares),
  !anyDuplicated(pares[c("consulta", "id")])
)

write.csv(pares, "pares_consulta_documento.csv",
          row.names = FALSE, fileEncoding = "UTF-8")

cat("Total de pares:", nrow(pares), "\n")

Total de pares: 1100 


### 2.2 Escores do cross encoder (slide 6)

Modelo `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`. Esta é a parte demorada (instalação e download do modelo). Se `ce_escores.csv` já existir na sessão, a célula é pulada.

In [6]:
if (file.exists("ce_escores.csv")) {
  cat("ce_escores.csv já existe: geração pulada.\n")
} else {
  python <- Sys.which("python3")
  if (!nzchar(python)) stop("Python 3 não encontrado no ambiente.")

  instalacao <- system2(python, c("-m", "pip", "install", "-q", "sentence-transformers"))
  if (instalacao != 0) stop("Não foi possível instalar sentence-transformers.")

  codigo_python <- c(
    "import pandas as pd",
    "from sentence_transformers import CrossEncoder",
    "",
    "dados = pd.read_csv('pares_consulta_documento.csv')",
    "",
    "modelo = CrossEncoder(",
    "    'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'",
    ")",
    "",
    "pares = list(zip(",
    "    dados['texto_consulta'].astype(str),",
    "    dados['texto_documento'].astype(str)",
    "))",
    "",
    "dados['escore'] = modelo.predict(",
    "    pares, batch_size=16, show_progress_bar=True",
    ")",
    "",
    "dados[['consulta', 'id', 'escore']].to_csv(",
    "    'ce_escores.csv', index=False",
    ")",
    "",
    "print('Arquivo ce_escores.csv gerado com sucesso!')"
  )

  writeLines(codigo_python, "gerar_escores.py")
  resultado <- system2(python, "gerar_escores.py")
  if (resultado != 0) stop("Falha na geração dos escores.")
  stopifnot(file.exists("ce_escores.csv"))
}

In [7]:
escores <- read.csv("ce_escores.csv", stringsAsFactors = FALSE)

stopifnot(
  nrow(escores) == nrow(pares),
  !anyNA(escores),
  !anyDuplicated(escores[c("consulta", "id")]),
  setequal(escores$consulta, names(consultas)),
  setequal(escores$id, names(docs))
)

cat("Linhas:", nrow(escores), "\n")
cat("Documentos:", length(unique(escores$id)), "\n")
cat("Consultas:", length(unique(escores$consulta)), "\n")

Linhas: 1100 
Documentos: 220 
Consultas: 5 


### 2.3 Reordenar o top k (slide 8)

In [8]:
reordenar <- function(rk, k, sc) {
  ids <- names(rk)
  k <- min(k, length(ids))
  if (k == 0) return(ids)
  top  <- ids[seq_len(k)]
  nova <- top[order(-sc[top], seq_along(top))]   # empate: ordem do BM25
  cauda <- if (k < length(ids)) ids[(k + 1):length(ids)] else character(0)
  c(nova, cauda)
}

ranking_rerank <- function(q, k) {
  bm25 <- ranking_bm25(consultas[[q]], ix, cfg)
  if (k == 0) return(names(bm25))

  tabela <- escores[escores$consulta == q, ]
  sc <- setNames(tabela$escore, tabela$id)
  stopifnot(!anyNA(sc[names(bm25)]))

  reordenar(bm25, k, sc)
}

cat("Função criada com sucesso!\n")

Função criada com sucesso!


### 2.4 Exemplo na q01, antes e depois, com k = 5 (slides 6 e 8)

In [9]:
q <- "q01"
k <- 5

antes  <- ranking_rerank(q, 0)
depois <- ranking_rerank(q, k)

comparacao <- data.frame(
  posicao = 1:5,
  BM25 = antes[1:5],
  BM25_Rerank = depois[1:5]
)

print(comparacao)

  posicao           BM25    BM25_Rerank
1       1 san_89a217bd1e san_89a217bd1e
2       2 san_b93b880263 san_787b827e51
3       3 san_31603f8e29 san_b93b880263
4       4 san_787b827e51  sv_758239684f
5       5  sv_758239684f san_31603f8e29


### 2.5 Avaliação no desenvolvimento (slides 7 e 8)

In [10]:
calcular_ap <- function(q, k) {
  metricas(ranking_rerank(q, k), qrels[[q]], cfg$limiar)[["AP"]]
}

avaliar <- function(consultas_avaliadas) {
  resultado <- sapply(ks, function(k) {
    sapply(consultas_avaliadas, function(q) calcular_ap(q, k))
  })
  colnames(resultado) <- paste0("k", ks)
  data.frame(consulta = consultas_avaliadas, resultado, row.names = NULL)
}

resultado_dev <- avaliar(dev)
print(resultado_dev)

map_dev <- colMeans(resultado_dev[-1])
print(round(map_dev, 3))

  consulta    k0    k5       k10       k20
1      q01 0.750 1.000 1.0000000 1.0000000
2      q02 0.225 0.225 0.2261905 0.1833333
3      q03 1.000 1.000 1.0000000 1.0000000
   k0    k5   k10   k20 
0.658 0.742 0.742 0.728 


### 2.6 Escolha do k só pelo desenvolvimento e teste uma vez (slides 7 e 8)

Regra: entre os valores de k com MAP até 0,005 abaixo do melhor, ficar com o menor. O teste só é calculado depois da escolha, para o k escolhido.

In [11]:
melhor_map <- max(map_dev)
cat("Maior MAP no desenvolvimento: k =", ks[which.max(map_dev)],
    "(", round(melhor_map, 4), ")\n")

candidatos <- which(map_dev >= melhor_map - 0.005)
melhor_k   <- min(ks[candidatos])

# teste, uma vez, só para o k escolhido
map_teste_k <- mean(sapply(teste, function(q) calcular_ap(q, melhor_k)))

cat("\nRerank escolhido:", melhor_k, "\n")
cat("MAP Dev:", round(map_dev[paste0("k", melhor_k)], 3),
    "  MAP Teste:", round(map_teste_k, 3), "\n")

Maior MAP no desenvolvimento: k = 10 ( 0.7421 )

Rerank escolhido: 5 
MAP Dev: 0.742   MAP Teste: 1 


## Parte 3 · Lorenzo: proximidade e janela

### 3.1 Função de proximidade e conferência à mão (slide 9)

In [12]:
# Para cada par de termos distintos da consulta, pega a MENOR distância entre
# as ocorrências no documento. Se a distância cabe na janela w, o par vale
# 1 / distância (colado = 1, um termo no meio = 1/2, ...). Senão vale 0.
# ordenada = TRUE: o par só conta se aparece na mesma ordem da consulta.
proximidade <- function(termos, tokens, w, ordenada = TRUE) {
  termos <- unique(termos)
  if (length(termos) < 2) return(0)
  pos <- lapply(termos, function(t) which(tokens == t))
  total <- 0
  for (i in 1:(length(termos) - 1)) {
    for (j in (i + 1):length(termos)) {
      pa <- pos[[i]]; pb <- pos[[j]]
      if (!length(pa) || !length(pb)) next
      d <- outer(pa, pb, function(a, b) b - a)          # distância com sinal
      d <- if (ordenada) d[d > 0] else abs(d)[d != 0]
      if (!length(d)) next
      dmin <- min(d)
      if (dmin <= w) total <- total + 1 / dmin
    }
  }
  total
}

# Conferência à mão: contas que sabemos de cabeça
tk <- c("modelo", "x", "de", "y", "de", "texto")
stopifnot(
  proximidade(c("modelo", "de"), tk, w = 1) == 0,
  proximidade(c("modelo", "de"), tk, w = 3) == 1/2,
  proximidade(c("modelo", "de", "texto"), tk, w = 5) == 1/2 + 1/5 + 1,
  proximidade(c("de", "modelo"), tk, w = 3, ordenada = TRUE)  == 0,
  proximidade(c("de", "modelo"), tk, w = 3, ordenada = FALSE) == 1/2
)
cat("Conferência à mão: OK\n")

Conferência à mão: OK


Exemplo do slide 8 (consulta primeira, cidade, Brasil, janela w = 3), conta feita pela própria função num trecho de exemplo:

In [13]:
trecho <- c("primeira", "cidade", "de", "brasil")
termos <- c("primeira", "cidade", "brasil")
cat("Escore do trecho de exemplo (w = 3):", round(proximidade(termos, trecho, w = 3), 2), "\n")

Escore do trecho de exemplo (w = 3): 1.83 


### 3.2 Reordenar o top k pela proximidade (slide 9)

A função `reordenar` é a mesma da Parte 2.

In [14]:
ranking_prox <- function(q, k, w, ordenada = TRUE) {
  bm25_rk <- ranking_bm25(consultas[[q]], ix, cfg)
  if (k == 0) return(names(bm25_rk))
  termos <- preparar(consultas[[q]], cfg)
  top <- names(bm25_rk)[seq_len(min(k, length(bm25_rk)))]
  sc <- sapply(top, function(d) proximidade(termos, ix$tokens[[d]], w, ordenada))
  reordenar(bm25_rk, k, sc)
}

ap_de <- function(q, k, w, ordenada = TRUE) {
  metricas(ranking_prox(q, k, w, ordenada), qrels[[q]], cfg$limiar)[["AP"]]
}

cat("Funções criadas com sucesso!\n")

Funções criadas com sucesso!


### 3.3 Exemplo na q02, antes e depois, com k = 5 e w = 3 (slide 10)

In [15]:
q <- "q02"
antes  <- ranking_prox(q, 0, 0)
depois <- ranking_prox(q, 5, 3, TRUE)   # top 5 do BM25, janela w = 3, ordem da consulta

marca <- function(ids) {
  g <- qrels[[q]][ids]
  paste0(ids, ifelse(!is.na(g) & g == 2, " *", ""))
}

print(data.frame(posicao = 1:5,
                 BM25 = marca(antes[1:5]),
                 BM25_Proximidade = marca(depois[1:5])),
      row.names = FALSE, right = FALSE)
cat("* trecho relevante\n")

 posicao BM25            BM25_Proximidade
 1       sv_6d5727fb12   sv_6d5727fb12   
 2       sv_651336a6c4   sv_df847bdd1e   
 3       sv_df847bdd1e   sv_651336a6c4   
 4       sv_70b5fb2a9e   sv_b8d2c80da5 * 
 5       sv_b8d2c80da5 * sv_70b5fb2a9e   
* trecho relevante


### 3.4 Grade no desenvolvimento, escolha e teste uma vez (slide 10)

In [16]:
grade <- expand.grid(k = c(0, 5, 10, 20), w = c(1, 2, 3, 5, 10),
                     ordenada = c(TRUE, FALSE))
grade <- grade[!(grade$k == 0 & (grade$w != 1 | !grade$ordenada)), ]  # k = 0 é o BM25 puro
grade$MAP_dev <- sapply(seq_len(nrow(grade)), function(i)
  mean(sapply(dev, function(q) ap_de(q, grade$k[i], grade$w[i], grade$ordenada[i]))))
grade <- grade[order(-grade$MAP_dev, grade$k, grade$w), ]

cat("Grade no desenvolvimento (topo):\n")
print(head(grade, 10), row.names = FALSE, digits = 4)

base_dev <- grade$MAP_dev[grade$k == 0]
cat("\nBM25 puro (k = 0), MAP no desenvolvimento:", round(base_dev, 4), "\n")

Grade no desenvolvimento (topo):
 k  w ordenada MAP_dev
 5  1     TRUE  0.6667
 5  1    FALSE  0.6667
 5  2     TRUE  0.6667
 5  2    FALSE  0.6667
 5  3     TRUE  0.6667
 5  3    FALSE  0.6667
 5  5     TRUE  0.6667
 5  5    FALSE  0.6667
 5 10     TRUE  0.6667
 5 10    FALSE  0.6667

BM25 puro (k = 0), MAP no desenvolvimento: 0.6583 


In [17]:
melhor <- max(grade$MAP_dev)
empatados <- grade[grade$MAP_dev >= melhor - 0.005, ]
escolha <- empatados[order(empatados$k, empatados$w, -empatados$ordenada), ][1, ]

if (escolha$MAP_dev <= base_dev + 1e-9) {
  cat("Sem vencedora: nenhuma versão superou o BM25 no desenvolvimento. Fica o BM25.\n")
  escolha <- grade[grade$k == 0, ][1, ]
}

# teste, uma vez só
map_teste <- function(k, w, ordenada)
  mean(sapply(teste, function(q) ap_de(q, k, w, ordenada)))

teste_bm25 <- map_teste(0, 1, TRUE)
teste_prox <- map_teste(escolha$k, escolha$w, escolha$ordenada)

cat("Escolha: k =", escolha$k, "| w =", escolha$w, "|",
    ifelse(escolha$ordenada, "ordenada", "livre"), "\n")
cat("MAP_dev =", round(escolha$MAP_dev, 4), "\n")
cat("Teste: BM25", round(teste_bm25, 4), "| Prox.", round(teste_prox, 4), "\n")

Escolha: k = 5 | w = 1 | ordenada 
MAP_dev = 0.6667 
Teste: BM25 0.7083 | Prox. 0.7083 


### 3.5 Resultado: MAP no desenvolvimento por k e w (slide 11)

Mesmo valor para a versão ordenada e a livre.

In [18]:
ws <- c(1, 2, 3, 5, 10)
tab <- sapply(ws, function(w)
  sapply(c(5, 10, 20), function(k)
    mean(sapply(dev, function(q) ap_de(q, k, w, TRUE)))))
dimnames(tab) <- list(paste0("k = ", c(5, 10, 20)), paste0("w = ", ws))

tab_livre <- sapply(ws, function(w)
  sapply(c(5, 10, 20), function(k)
    mean(sapply(dev, function(q) ap_de(q, k, w, FALSE)))))

cat("BM25 puro (k = 0):", round(base_dev, 3), "\n\n")
print(round(tab, 3))
cat("\nOrdenada e livre iguais:", isTRUE(all.equal(tab, tab_livre, check.attributes = FALSE)), "\n")

cat("\nAP por consulta no desenvolvimento, BM25 e proximidade (k = 5, w = 1):\n")
print(data.frame(consulta = dev,
                 BM25 = sapply(dev, function(q) ap_de(q, 0, 1)),
                 Proximidade = sapply(dev, function(q) ap_de(q, 5, 1, TRUE))),
      row.names = FALSE, digits = 4)

BM25 puro (k = 0): 0.658 

       w = 1 w = 2 w = 3 w = 5 w = 10
k = 5  0.667 0.667 0.667 0.667  0.667
k = 10 0.662 0.662 0.658 0.658  0.658
k = 20 0.653 0.653 0.651 0.651  0.651

Ordenada e livre iguais: TRUE 

AP por consulta no desenvolvimento, BM25 e proximidade (k = 5, w = 1):
 consulta  BM25 Proximidade
      q01 0.750        0.75
      q02 0.225        0.25
      q03 1.000        1.00


In [19]:
if (escolha$k > 0) {
  efeito_w <- data.frame(
    w = ws,
    MAP_dev = sapply(ws, function(w)
      mean(sapply(dev, function(q) ap_de(q, escolha$k, w, escolha$ordenada)))))
  cat("Efeito da janela (k e ordem da escolha), MAP no desenvolvimento:\n")
  print(efeito_w, row.names = FALSE, digits = 4)
  if (length(unique(round(efeito_w$MAP_dev, 10))) == 1)
    cat("A janela w não muda o resultado: registrar como janela sem efeito.\n")

  mexeu <- sapply(c(dev, teste), function(q) {
    a <- ranking_prox(q, 0, 1)[seq_len(escolha$k)]
    b <- ranking_prox(q, escolha$k, escolha$w, escolha$ordenada)[seq_len(escolha$k)]
    sum(a != b)
  })
  cat("\nPosições trocadas no top", escolha$k, "por consulta:\n")
  print(mexeu)
}

Efeito da janela (k e ordem da escolha), MAP no desenvolvimento:
  w MAP_dev
  1  0.6667
  2  0.6667
  3  0.6667
  5  0.6667
 10  0.6667
A janela w não muda o resultado: registrar como janela sem efeito.

Posições trocadas no top 5 por consulta:
q01 q02 q03 q04 q05 
  0   4   3   0   0 


## Comparação final (slide 12)

In [20]:
comparacao_final <- data.frame(
  Metodo = c("BM25 puro",
             paste0("Cross encoder (k = ", melhor_k, ")"),
             paste0("Proximidade (k = ", escolha$k, ", w = ", escolha$w, ")")),
  Dev   = round(c(map_bm25_dev, map_dev[paste0("k", melhor_k)], escolha$MAP_dev), 3),
  Teste = round(c(map_bm25_teste, map_teste_k, teste_prox), 3)
)
print(comparacao_final, row.names = FALSE)

cat("\nCuidado: são só", length(consultas), "consultas e",
    sum(julgamentos$grau == 2), "trechos relevantes, então as diferenças são frágeis.\n")

                     Metodo   Dev Teste
                  BM25 puro 0.658 0.708
      Cross encoder (k = 5) 0.742 1.000
 Proximidade (k = 5, w = 1) 0.667 0.708

Cuidado: são só 5 consultas e 9 trechos relevantes, então as diferenças são frágeis.
